In [2]:
import json
from io import BytesIO
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
import boto3
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

CLASS_NAMES = ["Small", "Medium", "Large"]
NORMALIZATION = {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]}


from dotenv import load_dotenv
from IPython.display import display

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
load_dotenv(PROJECT_ROOT / ".env")


True

In [3]:
#local training settings and s3 paths
EPOCHS = 100
BATCH_SIZE = 16
LEARNING_RATE = 0.001
AWS_PROFILE = os.environ.get("AWS_PROFILE", "ATLAS")
AWS_REGION = "us-east-2"
BUCKET = "atlas-acm-research-f26"
IMAGE_PREFIX = "satellite/processed/"
METADATA_KEY = "dataset/im3_atlas_cleaned.csv"
OUTPUT_PREFIX = "artifacts/baseline/"

#all data and results stay in memory until results are uploaded to S3.
job_name = datetime.now(timezone.utc).strftime("local-%Y%m%d-%H%M%S-%f")
s3_output = f"{OUTPUT_PREFIX.rstrip('/')}/{job_name}/"

torch.manual_seed(42)
print(f"Python: {sys.executable}")
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is required. Select the atlas .venv kernel with CUDA-enabled PyTorch."
    )
device = torch.device("cuda")
print(f"PyTorch: {torch.__version__} | CUDA build: {torch.version.cuda}")
print(f"Training GPU: {torch.cuda.get_device_name(0)}")
    
available_profiles = boto3.Session().available_profiles
aws_session = boto3.Session(
    profile_name=AWS_PROFILE if AWS_PROFILE in available_profiles else None,
    region_name=AWS_REGION,
)
if aws_session.get_credentials() is None:
    raise RuntimeError(
        "AWS credentials are missing. Configure the ATLAS profile with "
        "aws login --profile ATLAS --region us-east-2" 
    )
s3 = aws_session.client("s3")
print(f"Run: {job_name} | Device: {device}")


Python: c:\Users\kevin\Projects\ATLAS\.venv\Scripts\python.exe
PyTorch: 2.11.0+cu128 | CUDA build: 12.8
Training GPU: NVIDIA GeForce RTX 5070
Run: local-20261026-033724-930916 | Device: cuda


In [4]:
#read CSV into RAM and list tensor keys
def read_s3_bytes(key):
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"]
    try:
        return body.read()
    finally:
        body.close()

metadata = pd.read_csv(BytesIO(read_s3_bytes(METADATA_KEY)), dtype={"id": str})
tensor_keys = []
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=IMAGE_PREFIX):
    tensor_keys.extend(item["Key"] for item in page.get("Contents", []) if item["Key"].endswith(".pt"))
if not tensor_keys:
    raise ValueError(f"No tensors found at s3://{BUCKET}/{IMAGE_PREFIX}")
tensor_cache = {}  #shared CPU RAM cache avoids fetching the same tensor each epoch.
print(f"Found {len(tensor_keys)} tensors in S3; metadata loaded into memory.")


Found 250 tensors in S3; metadata loaded into memory.


In [5]:
#baseline dataset reads tensors from S3 into RAM
class DataCenterDataset(Dataset):
    def __init__(self, frame, augment=False):
        self.rows = list(frame[["s3_key", "size_label"]].itertuples(index=False, name=None))
        
        #rotations and flips to help with overfitting
        self.augment = augment
        self.flips = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5)
        ])
        
        self.normalize = transforms.Normalize(**NORMALIZATION)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        key, label = self.rows[index]
        if key not in tensor_cache:
            data = torch.load(BytesIO(read_s3_bytes(key)), map_location="cpu", weights_only=True)
            tensor_cache[key] = data["image"].float()
        
        image = tensor_cache[key].clone()
        if self.augment:
            #randomly rotate by 0, 90, 180, 270 degrees
            k = torch.randint(0,4,()).item()
            image = torch.rot90(image,k,dims=(-2,-1))
            image = self.flips(image)
            
        return self.normalize(image), int(label)
    
def normalize_id(value):
    if pd.isna(value):
        return None
    value = str(value).strip()
    return str(int(value)) if value.isdecimal() else value

def load_and_split_data(tensor_keys, metadata):
    metadata = metadata.copy()
    metadata["id"] = metadata["id"].map(normalize_id)

    metadata["sqft"] = pd.to_numeric(metadata["sqft"], errors="coerce")
    metadata = metadata[metadata["sqft"] > 0].copy()

    q1, q3 = metadata["sqft"].quantile([0.25, 0.75])
    metadata["size_label"] = metadata["sqft"].apply(
        lambda x: 0 if x < q1 else 1 if x <= q3 else 2
    )

    records = [
        {
            "id": normalize_id(Path(key).stem),
            "s3_key": key,
        }
        for key in tensor_keys
    ]   

    images = pd.DataFrame(records).merge(
        metadata[["id", "size_label"]],
        on="id",
        validate="many_to_one",
    )

    print(
        f"Matched {len(images)} tensors "
        f"across {images['id'].nunique()} facilities"
    )

    facilities = images[["id", "size_label"]].drop_duplicates()

    train, remainder = train_test_split(
        facilities,
        test_size=0.30,
        random_state=42,
        stratify=facilities["size_label"]
    )

    validation, test = train_test_split(
        remainder,
        test_size=0.50,
        random_state=42,
        stratify=remainder["size_label"]
    )

    frames = [
        images[images["id"].isin(split["id"])].copy()
        for split in (train, validation, test)
    ]

    return frames, {"q1": float(q1), "q3": float(q3)}

frames, thresholds = load_and_split_data(tensor_keys, metadata)
loaders = [
    DataLoader(
        DataCenterDataset(frame, augment=(index == 0)),
        batch_size=BATCH_SIZE,
        shuffle=(index == 0),
        num_workers=0,
    )
    for index, frame in enumerate(frames)
]
for name, frame in zip(("train", "validation", "test"), frames):
    print(f"{name}: {len(frame)} tensors, {frame['id'].nunique()} facilities")
print("Square-footage thresholds:", thresholds)


Matched 249 tensors across 245 facilities
train: 174 tensors, 171 facilities
validation: 38 tensors, 37 facilities
test: 37 tensors, 37 facilities
Square-footage thresholds: {'q1': 58043.5, 'q3': 231044.0}


In [6]:
def run_epoch(model, loader, criterion, device, optimizer=None):
    model.eval()
    if optimizer is not None:
        model.fc.train()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(optimizer is not None):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            #clear the previous batch's gradients before calculating new ones.
            if optimizer is not None:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            #calculate gradients and use them to update the trainable weights.
            if optimizer is not None:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

#reuse pretrained image features and train a new three-class classification head.
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for parameter in model.parameters():
    parameter.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
model.to(device)

#cross-entropy measures classification error; Adam updates only the new head.
class_counts = (
    frames[0]["size_label"]
    .value_counts()
    .reindex(range(len(CLASS_NAMES)), fill_value=0)
)

class_weights = torch.tensor(
    len(frames[0]) / (len(CLASS_NAMES) * class_counts.to_numpy()),
    dtype=torch.float32,
    device=device,
)

criterion = nn.CrossEntropyLoss(weight=class_weights)
print("Class weights:", class_weights.cpu().tolist())
    
optimizer = torch.optim.Adam(model.fc.parameters(), lr=LEARNING_RATE)

history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_accuracy = run_epoch(
        model, loaders[0], criterion, device, optimizer
    )
    val_loss, val_accuracy = run_epoch(
        model, loaders[1], criterion, device
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
    })

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.2%} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.2%}",
        flush=True,
    )


Class weights: [1.8125, 0.6373626589775085, 1.1372549533843994]
Epoch 1/100 | Train Loss: 1.2199 | Train Acc: 39.08% | Val Loss: 1.1781 | Val Acc: 23.68%
Epoch 2/100 | Train Loss: 1.1199 | Train Acc: 35.06% | Val Loss: 1.1622 | Val Acc: 31.58%
Epoch 3/100 | Train Loss: 1.0711 | Train Acc: 48.85% | Val Loss: 1.1292 | Val Acc: 31.58%
Epoch 4/100 | Train Loss: 1.0387 | Train Acc: 47.70% | Val Loss: 1.1257 | Val Acc: 39.47%
Epoch 5/100 | Train Loss: 1.0067 | Train Acc: 54.02% | Val Loss: 1.1012 | Val Acc: 31.58%
Epoch 6/100 | Train Loss: 0.9479 | Train Acc: 51.15% | Val Loss: 1.0900 | Val Acc: 36.84%
Epoch 7/100 | Train Loss: 0.9762 | Train Acc: 58.05% | Val Loss: 1.1247 | Val Acc: 44.74%
Epoch 8/100 | Train Loss: 0.9192 | Train Acc: 51.15% | Val Loss: 1.0853 | Val Acc: 36.84%
Epoch 9/100 | Train Loss: 0.9138 | Train Acc: 54.60% | Val Loss: 1.0794 | Val Acc: 47.37%
Epoch 10/100 | Train Loss: 0.8887 | Train Acc: 60.34% | Val Loss: 1.0645 | Val Acc: 36.84%
Epoch 11/100 | Train Loss: 0.8609 |

In [7]:
#produce metrics and plot in RAM
def figure_bytes(figure):
    buffer = BytesIO()
    figure.savefig(buffer, format="png", dpi=150)
    return buffer.getvalue()

def save_evaluation(model, loader, device, history, results):
    predictions, actual = [], []
    model.eval()
    with torch.no_grad():
        for images, labels in loader:
            predictions.extend(model(images.to(device)).argmax(1).cpu().tolist())
            actual.extend(labels.tolist())
    #rows are actual classes, columns are predicted classes.
    matrix = confusion_matrix(actual, predictions, labels=[0, 1, 2])
    report_args = dict(labels=[0, 1, 2], target_names=CLASS_NAMES, zero_division=0)
    report = classification_report(actual, predictions, **report_args)
    metrics = {
        "test_accuracy": float(accuracy_score(actual, predictions)),
        "test_macro_f1": float(
            f1_score(
                actual,
                predictions,
                labels=[0, 1, 2],
                average="macro",
                zero_division=0,
            )
        ),
        "classification_report": classification_report(actual, predictions, output_dict=True, **report_args),
        "confusion_matrix": matrix.tolist(),
    }
    results["metrics.json"] = json.dumps(metrics, indent=2).encode("utf-8")
    results["classification_report.txt"] = report.encode("utf-8")
    results["training_history.csv"] = pd.DataFrame(history).to_csv(index=False).encode("utf-8")
    print(f"Test accuracy: {metrics['test_accuracy']:.4f} | Macro F1: {metrics['test_macro_f1']:.4f}")
    print(report, flush=True)

    #compare training and validation over time to show learning and possible overfitting.
    for metric in ("loss", "accuracy"):
        fig, ax = plt.subplots()
        for split, label in (("train", "Training"), ("val", "Validation")):
            ax.plot([row["epoch"] for row in history],
                    [row[f"{split}_{metric}"] for row in history], label=label)
        ax.set(xlabel="Epoch", ylabel=metric.capitalize(),
               title=f"Training and validation {metric}")
        if metric == "accuracy":
            ax.set_ylim(0, 1)
        ax.legend()
        fig.tight_layout()
        results[f"{metric}_curve.png"] = figure_bytes(fig)
        plt.close(fig)
    display = ConfusionMatrixDisplay(matrix, display_labels=CLASS_NAMES)
    display.plot(cmap="Blues", values_format="d")
    display.ax_.set_title("Test confusion matrix")
    display.figure_.tight_layout()
    results["confusion_matrix.png"] = figure_bytes(display.figure_)
    plt.close(display.figure_)

results = {}
save_evaluation(model, loaders[2], device, history, results)
results["data_splits.csv"] = pd.concat([
    frame.assign(split=name)
    for name, frame in zip(("train", "validation", "test"), frames)
]).to_csv(index=False).encode("utf-8")
#save trained weights in RAM, the upload cell sends this under the current job.
buffer = BytesIO()
torch.save({name: value.detach().cpu() for name, value in model.state_dict().items()}, buffer)
results["model.pkl"] = buffer.getvalue()
print("Results prepared in memory:", ", ".join(sorted(results)))


Test accuracy: 0.4865 | Macro F1: 0.4534
              precision    recall  f1-score   support

       Small       0.30      0.50      0.38         6
      Medium       0.58      0.55      0.56        20
       Large       0.50      0.36      0.42        11

    accuracy                           0.49        37
   macro avg       0.46      0.47      0.45        37
weighted avg       0.51      0.49      0.49        37

Results prepared in memory: accuracy_curve.png, classification_report.txt, confusion_matrix.png, data_splits.csv, loss_curve.png, metrics.json, model.pkl, training_history.csv


In [ ]:
#gradcam - side by side heatmap with output class 

model.eval()
target_layer = model.layer4[-1]


def make_gradcam(image, target_class):
    captured = {}

    def capture_features(module, inputs, output):
        captured["features"] = output

    handle = target_layer.register_forward_hook(capture_features)

    try:
        #input gradients allow Grad-CAM to work with the frozen backbone.
        with torch.enable_grad():
            x = image.unsqueeze(0).to(device).detach()
            x.requires_grad_(True)

            logits = model(x)
            features = captured["features"]

            gradients = torch.autograd.grad(
                logits[0, target_class],
                features,
            )[0]

            weights = gradients.mean(dim=(2, 3), keepdim=True)
            cam = (weights * features).sum(dim=1, keepdim=True)
            cam = F.relu(cam)

            cam = F.interpolate(
                cam,
                size=image.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )

            cam = cam.detach().squeeze(0).squeeze(0)
            maximum = cam.max()

            if maximum > 0:
                cam = cam / maximum
            else:
                print("No positive Grad-CAM signal for this sample/class.")

            return cam.cpu().numpy()

    finally:
        #avoid accumulating hooks when this cell is rerun.
        handle.remove()


#test images have normalization but no random augmentation.
test_dataset = loaders[2].dataset
if getattr(test_dataset, "augment", False):
    raise RuntimeError("Disable augmentation for the test dataset.")

selected = {}

with torch.no_grad():
    for index in range(len(test_dataset)):
        image, actual = test_dataset[index]
        probabilities = model(
            image.unsqueeze(0).to(device)
        ).softmax(dim=1)[0].cpu()

        predicted = int(probabilities.argmax().item())
        outcome = "correct" if predicted == actual else "incorrect"

        if outcome not in selected:
            row = frames[2].iloc[index]

            selected[outcome] = {
                "image": image,
                "id": str(row["id"]),
                "s3_key": row["s3_key"],
                "actual": int(actual),
                "predicted": predicted,
                "probabilities": probabilities.tolist(),
            }

        if len(selected) == 2:
            break


sample_details = []

for outcome in ("correct", "incorrect"):
    if outcome not in selected:
        print(f"No {outcome} prediction found in the test set.")
        continue

    sample = selected[outcome]
    predicted = sample["predicted"]
    actual = sample["actual"]

    #explains the predicted class's raw score.
    heatmap = make_gradcam(sample["image"], predicted)

    #display the original RGB tensor, before normalization.
    original = (
        tensor_cache[sample["s3_key"]]
        .detach()
        .cpu()
        .permute(1, 2, 0)
        .clamp(0, 1)
        .numpy()
    )

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    axes[0].imshow(original)
    axes[0].set_title("Original satellite image")

    axes[1].imshow(original)
    overlay = axes[1].imshow(
        heatmap,
        cmap="jet",
        alpha=0.45,
        vmin=0,
        vmax=1,
    )
    axes[1].set_title(
        f"Grad-CAM for predicted class: {CLASS_NAMES[predicted]}"
    )
    fig.colorbar(overlay, ax=axes[1], fraction=0.046, pad=0.04)

    for axis in axes:
        axis.axis("off")

    probability = sample["probabilities"][predicted]
    fig.suptitle(
        f"{outcome.upper()} | Facility {sample['id']}\n"
        f"Actual: {CLASS_NAMES[actual]} | "
        f"Predicted: {CLASS_NAMES[predicted]} "
        f"({probability:.1%})"
    )
    fig.tight_layout()

    name = f"gradcam/{outcome}_{sample['id']}.png"
    results[name] = figure_bytes(fig)

    display(fig)
    plt.close(fig)

    sample_details.append({
        "outcome": outcome,
        "facility_id": sample["id"],
        "s3_key": sample["s3_key"],
        "actual_class": CLASS_NAMES[actual],
        "predicted_class": CLASS_NAMES[predicted],
        "class_probabilities": dict(
            zip(CLASS_NAMES, sample["probabilities"])
        ),
        "target_layer": "layer4[-1]",
        "image_file": name,
    })

results["gradcam/samples.json"] = json.dumps(
    {"job_name": job_name, "samples": sample_details},
    indent=2,
).encode("utf-8")

print("Grad-CAM images and sample details are prepared in RAM.")

<Figure size 1000x500 with 3 Axes>

<Figure size 1000x500 with 3 Axes>

Grad-CAM images and sample details are prepared in RAM.


In [9]:
#upload to s3 from RAM
if "model.pkl" not in results:
    raise RuntimeError("Run the evaluation cell to prepare model.pkl before uploading.")
content_types = {".png": "image/png", ".json": "application/json",
                 ".csv": "text/csv", ".txt": "text/plain", ".pkl": "application/octet-stream"}
for name, payload in sorted(results.items()):
    s3.upload_fileobj(BytesIO(payload), BUCKET, f"{s3_output}{name}",
                      ExtraArgs={"ContentType": content_types[Path(name).suffix]})
    print(f"Uploaded: {name}")
print(f"Metrics and visuals: s3://{BUCKET}/{s3_output}")


Uploaded: accuracy_curve.png
Uploaded: classification_report.txt
Uploaded: confusion_matrix.png
Uploaded: data_splits.csv
Uploaded: gradcam/correct_977653854.png
Uploaded: gradcam/incorrect_283051477.png
Uploaded: gradcam/samples.json
Uploaded: loss_curve.png
Uploaded: metrics.json
Uploaded: model.pkl
Uploaded: training_history.csv
Metrics and visuals: s3://atlas-acm-research-f26/artifacts/baseline/local-20261026-033724-930916/
